# 01 · Data collection
Genuine data only: OpenAQ public archive (CPCB/MPCB/IITM stations, 15-min), Open-Meteo ERA5 weather, CAMS (benchmark only).

Re-run the full download with `python -m airsense.ml.data_collection --discover --start 2025-02-01`. This notebook inspects what was collected.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / "backend"))
import pandas as pd, numpy as np, json
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [2]:
from airsense.ml.data_collection import load_station_catalog
cat = pd.DataFrame([s.__dict__ for s in load_station_catalog()]); cat

,location_id,name,latitude,longitude,provider,city,first_seen
0,3409331,"Bhosari, Pune - IITM",18.640051,73.848956,IITM,Pune,20250225
1,3409435,"Gavalinagar, Pimpri Chinchwad - MPCB",18.636730,73.824870,MPCB,Pimpri-Chinchwad,20250219
2,3409436,"Park Street Wakad, Pimpri Chinchwad - MPCB",18.590510,73.779460,MPCB,Pimpri-Chinchwad,20250219
3,3409437,"Thergaon, Pimpri Chinchwad - MPCB",18.616318,73.765797,MPCB,Pimpri-Chinchwad,20250219
4,3409438,"Katraj Dairy, Pune - MPCB",18.454450,73.854155,MPCB,Pune,20250220
5,3409439,"Savitribai Phule Pune University, Pune - MPCB",18.547056,73.826908,MPCB,Pune,20250219
6,3409523,"Bhumkar Nagar, Pune - IITM",18.605770,73.749976,IITM,Pune,20250219
7,3409526,"Panchawati_Pashan, Pune - IITM",18.536457,73.805454,IITM,Pune,20250219
8,3409528,"Savta Mali Nagar, Pimpri-Chinchwad - IITM",18.614767,73.799516,IITM,Pimpri-Chinchwad,20250219
9,3410005,"Dhankawadi, Pune - IITM",18.459883,73.852193,IITM,Pune,20250219


In [3]:
man = pd.read_json(ROOT/'data/raw/manifest.jsonl', lines=True)
man.groupby('source').agg(entries=('source','size'), files=('downloaded','sum') if 'downloaded' in man else ('source','size'))

,entries,files
source,,
open-meteo-archive,80,0.0
open-meteo-cams,80,0.0
open-meteo-forecast,20,0.0
openaq-archive,30,5658.0


In [4]:
from airsense.ml.data_collection import read_openaq_raw
raw = read_openaq_raw(3409438)
print(raw.shape); raw.groupby(['parameter','units']).value.describe().round(2)

(434235, 10)


,,count,mean,std,min,25%,50%,75%,max
parameter,units,,,,,,,,
co,ppb,40282.0,0.79,0.50,-1.02,0.48,0.73,1.00,6.02
no,ppb,39597.0,6.97,17.97,-1885.62,2.85,5.97,9.30,605.86
no2,ppb,39989.0,18.20,16.51,-61.44,7.43,12.74,27.36,738.34
nox,ppb,23938.0,0.01,0.02,-1.13,0.01,0.01,0.02,0.45
o3,µg/m³,40387.0,29.49,28.94,-2270.96,10.04,25.72,40.43,1702.61
pm10,µg/m³,39796.0,95.62,2029.59,0.00,32.60,52.84,94.70,339070.41
pm25,µg/m³,40277.0,31.54,339.02,0.00,12.40,21.95,39.21,36216.20
relativehumidity,%,40678.0,60.20,17.53,0.00,45.60,62.68,76.22,85.70
so2,ppb,40143.0,3.65,19.59,-547.10,2.03,2.47,3.09,474.22


**Observation:** the raw archive contains physically impossible values (e.g. PM2.5 in the tens of thousands µg/m³) and inconsistent unit labels for gases - handled in preprocessing.